# S6E5 LB-blend menu

Builds 10 different blends of the saved `score*.csv` submissions and writes them as `blend01.csv` ... `blend10.csv`

Blend01: .94772
Blend02: .94848
Blend03: .94911
Blend04: .94880
Blend05: .94879
Blend06: .94898
Blend07: .94928
Blend08: .94928
Blend09: .94917
Blend10: .94...(ran out of submissions)

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

FOLDER = r'.'
ID_COL = 'id'
TARGET_COL = 'PitNextLap'

SOURCES = {
    'a_95025': 'score95025.csv',
    'b_95025': 'score95025(1).csv',
    'c_94990': 'score94990.csv',
    'd_94853': 'score94853.csv',
    'e_94970': 'score94970.csv',
    'f_94968': 'score94968.csv',
    'g_94966': 'score94966.csv',
}

preds = {}
ids = None
for k, fn in SOURCES.items():
    path = os.path.join(FOLDER, fn)
    df = pd.read_csv(path)
    if ids is None:
        ids = df[ID_COL].values
    else:
        assert (df[ID_COL].values == ids).all(), f'id mismatch in {fn}'
    preds[k] = df[TARGET_COL].values.astype(np.float64)
print('Loaded', len(preds), 'sources, rows per file:', len(ids))

Loaded 7 sources, rows per file: 188165


In [2]:
print('Spearman rank-correlation among sources (AUC sees only ranks):')
keys = list(preds)
header = '         ' + ' '.join(f'{k:>9}' for k in keys)
print(header)
for i, k in enumerate(keys):
    cells = [f'{k:>9}']
    for j, k2 in enumerate(keys):
        if j < i:
            cells.append('         ')
        else:
            r = spearmanr(preds[k], preds[k2]).statistic
            cells.append(f'{r:>9.4f}')
    print(' '.join(cells))

Spearman rank-correlation among sources (AUC sees only ranks):
           a_95025   b_95025   c_94990   d_94853   e_94970   f_94968   g_94966
  a_95025    1.0000    0.9649    0.9977    0.9841    0.9974    0.9981    0.9982
  b_95025              1.0000    0.9676    0.9752    0.9656    0.9649    0.9652
  c_94990                        1.0000    0.9883    0.9966    0.9965    0.9966
  d_94853                                  1.0000    0.9856    0.9842    0.9841
  e_94970                                            1.0000    0.9995    0.9994
  f_94968                                                      1.0000    0.9999
  g_94966                                                                1.0000


In [3]:
def to_rank(p):
    return pd.Series(p).rank(method='average', pct=True).to_numpy()

def rank_blend(members, weights=None):
    if weights is None:
        weights = [1.0] * len(members)
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    out = np.zeros(len(next(iter(members))))
    for wi, p in zip(w, members):
        out += wi * to_rank(p)
    return out

def prob_blend(members, weights=None):
    if weights is None:
        weights = [1.0] * len(members)
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    return sum(wi * p for wi, p in zip(w, members))

def logit_blend(members, weights=None, eps=1e-6):
    if weights is None:
        weights = [1.0] * len(members)
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    z = np.zeros(len(next(iter(members))))
    for wi, p in zip(w, members):
        pc = np.clip(p, eps, 1 - eps)
        z += wi * np.log(pc / (1.0 - pc))
    return 1.0 / (1.0 + np.exp(-z))

def geom_blend(members, weights=None, eps=1e-6):
    if weights is None:
        weights = [1.0] * len(members)
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    z = np.zeros(len(next(iter(members))))
    for wi, p in zip(w, members):
        pc = np.clip(p, eps, 1 - eps)
        z += wi * np.log(pc)
    return np.exp(z)

def save_blend(filename, pred):
    sub = pd.DataFrame({ID_COL: ids, TARGET_COL: pred})
    sub.to_csv(filename, index=False)
    return sub

In [4]:
a = preds['a_95025']
b = preds['b_95025']
c = preds['c_94990']
d = preds['d_94853']
e = preds['e_94970']
f = preds['f_94968']
g = preds['g_94966']

blends = []

blends.append(('blend01.csv',
               'rank-mean of the two .95025 submissions (a, b) - core hypothesis',
               rank_blend([a, b])))

blends.append(('blend02.csv',
               'rank-mean of (a, b) weighted 0.6 / 0.4 toward a',
               rank_blend([a, b], [0.6, 0.4])))

blends.append(('blend03.csv',
               'rank-mean of (a, b) weighted 0.7 / 0.3 toward a (safe-anchor variant)',
               rank_blend([a, b], [0.7, 0.3])))

blends.append(('blend04.csv',
               'rank-mean 3-way (a, b, c=94990) equal weight - adds best-of-second-tier',
               rank_blend([a, b, c])))

blends.append(('blend05.csv',
               'rank-mean 3-way (a, b, c) LB-weighted by (LB - 0.5)',
               rank_blend([a, b, c], [0.45025, 0.45025, 0.44990])))

blends.append(('blend06.csv',
               'rank-mean 4-way (a, b, c, d=94853) - d is decorrelated 2nd-tier diversifier',
               rank_blend([a, b, c, d])))

blends.append(('blend07.csv',
               'rank-mean 5-way (a, b, c, d, e=94970) - max diversity from top distinct families',
               rank_blend([a, b, c, d, e])))

blends.append(('blend08.csv',
               'rank-mean 5-way LB-weighted (a, b, c, d, e)',
               rank_blend([a, b, c, d, e], [0.45025, 0.45025, 0.44990, 0.44853, 0.44970])))

blends.append(('blend09.csv',
               'raw-probability mean of (a, b, c) - alt aggregator, sensitive to scale',
               prob_blend([a, b, c])))

blends.append(('blend10.csv',
               'logit-mean of (a, b, c) - sharpens disagreement; alt mechanism',
               logit_blend([a, b, c])))

for fname, desc, pred in blends:
    save_blend(os.path.join(FOLDER, fname), pred)
    r_a = spearmanr(pred, a).statistic
    r_b = spearmanr(pred, b).statistic
    print(f'{fname}  mean={pred.mean():.4f}  std={pred.std():.4f}  rho_a={r_a:.4f}  rho_b={r_b:.4f}  | {desc}')

blend01.csv  mean=0.5000  std=0.2861  rho_a=0.9911  rho_b=0.9910  | rank-mean of the two .95025 submissions (a, b) - core hypothesis
blend02.csv  mean=0.5000  std=0.2862  rho_a=0.9943  rho_b=0.9870  | rank-mean of (a, b) weighted 0.6 / 0.4 toward a
blend03.csv  mean=0.5000  std=0.2865  rho_a=0.9969  rho_b=0.9823  | rank-mean of (a, b) weighted 0.7 / 0.3 toward a (safe-anchor variant)
blend04.csv  mean=0.5000  std=0.2864  rho_a=0.9953  rho_b=0.9849  | rank-mean 3-way (a, b, c=94990) equal weight - adds best-of-second-tier
blend05.csv  mean=0.5000  std=0.2864  rho_a=0.9953  rho_b=0.9849  | rank-mean 3-way (a, b, c) LB-weighted by (LB - 0.5)
blend06.csv  mean=0.5000  std=0.2865  rho_a=0.9943  rho_b=0.9842  | rank-mean 4-way (a, b, c, d=94853) - d is decorrelated 2nd-tier diversifier
blend07.csv  mean=0.5000  std=0.2866  rho_a=0.9959  rho_b=0.9813  | rank-mean 5-way (a, b, c, d, e=94970) - max diversity from top distinct families
blend08.csv  mean=0.5000  std=0.2866  rho_a=0.9959  rho_b=0.